In [33]:
import tensorflow as tf
tf.keras.backend.clear_session()

import os
import numpy as np

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras import layers, models

In [34]:
BASE_DIR = r"C:/Users/raksh/x-ai for medical imaging/data/dental_xray"

TRAIN_DIR = os.path.join(BASE_DIR, "train")
VAL_DIR   = os.path.join(BASE_DIR, "val")
TEST_DIR  = os.path.join(BASE_DIR, "test")

MODEL_SAVE_PATH = "backend/saved_models/dental_binary_model.keras"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
EPOCHS = 20

In [35]:
def crop_center(img):
    h, w, _ = img.shape
    return img[int(h*0.3):int(h*0.8), int(w*0.2):int(w*0.8)]

def preprocess(img):
    img = crop_center(img)
    img = tf.image.resize(img, IMG_SIZE)
    img = preprocess_input(img)
    return img

In [41]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess,
    rotation_range=15,
    zoom_range=0.2,
    width_shift_range=0.1,
    height_shift_range=0.1,
    brightness_range=[0.7, 1.3],
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess
)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(224, 224),   # FIXED
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(224, 224),   # FIXED
    batch_size=BATCH_SIZE,
    class_mode="binary"
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(224, 224),   # FIXED
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class mapping:", train_gen.class_indices)

Found 168 images belonging to 2 classes.
Found 35 images belonging to 2 classes.
Found 36 images belonging to 2 classes.
Class mapping: {'CAVITY': 0, 'NORMAL': 1}


In [42]:
base_model = MobileNetV2(
    include_top=False,
    weights="imagenet",
    input_shape=(224,224,3)
)

base_model.trainable = False

inputs = layers.Input(shape=(224,224,3))
x = base_model(inputs, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)

x = layers.Dense(128, activation="relu")(x)
x = layers.Dropout(0.5)(x)

outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,427,201 (9.26 MB)

 Trainable params: 166,657 (651.00 KB)

 Non-trainable params: 2,260,544 (8.62 MB)

In [43]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [44]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

In [45]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 29s 1s/step - accuracy: 0.5238 - loss: 0.8892 - val_accuracy: 0.4571 - val_loss: 0.8145 - learning_rate: 1.0000e-04
Epoch 2/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 658ms/step - accuracy: 0.4881 - loss: 0.9691 - val_accuracy: 0.4571 - val_loss: 0.7873 - learning_rate: 1.0000e-04
Epoch 3/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 571ms/step - accuracy: 0.5357 - loss: 0.9633 - val_accuracy: 0.4571 - val_loss: 0.7538 - learning_rate: 1.0000e-04
Epoch 4/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 8s 711ms/step - accuracy: 0.4881 - loss: 1.0319 - val_accuracy: 0.5143 - val_loss: 0.7405 - learning_rate: 1.0000e-04
Epoch 5/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 13s 938ms/step - accuracy: 0.5655 - loss: 0.8129 - val_accuracy: 0.5429 - val_loss: 0.7357 - learning_rate: 1.0000e-04
Epoch 6/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 7s 591ms/step - accuracy: 0.5833 - loss: 0.7937 - val_accuracy: 0.4857 - val_loss: 0.7414 - learning_rate: 1.0000e-04
Epoch 7/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 6s 566ms/step - accuracy: 0

In [46]:
test_loss, test_acc = model.evaluate(test_gen)
print("✅ Test Accuracy:", test_acc)

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 295ms/step - accuracy: 0.4167 - loss: 0.7882
✅ Test Accuracy: 0.4166666567325592


In [48]:
x_batch, y_batch = next(test_gen)

preds = model.predict(x_batch)

pred_labels = (preds > 0.5).astype(int).flatten()
true_labels = y_batch.astype(int)

print("Pred:", pred_labels[:10])
print("True:", true_labels[:10])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 326ms/step
Pred: [1 1 0 0 0 1 0 1 0 0]
True: [0 0 1 1 1 1 1 1 1 1]


In [22]:
base_model.trainable = True

for layer in base_model.layers[:-70]:   # partial unfreeze
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [23]:
callbacks_ft = [
    tf.keras.callbacks.ModelCheckpoint(
        MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.3,
        patience=3,
        min_lr=1e-6
    )
]

In [24]:
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS_PHASE2,
    class_weight=class_weights,
    callbacks=callbacks_ft
)

Epoch 1/12
11/11 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.5893 - loss: 1.0201 - val_accuracy: 0.5667 - val_loss: 0.7426 - learning_rate: 1.0000e-05
Epoch 2/12
11/11 ━━━━━━━━━━━━━━━━━━━━ 14s 1s/step - accuracy: 0.5417 - loss: 0.9301 - val_accuracy: 0.5333 - val_loss: 0.7317 - learning_rate: 1.0000e-05
Epoch 3/12
11/11 ━━━━━━━━━━━━━━━━━━━━ 10s 875ms/step - accuracy: 0.5952 - loss: 0.9887 - val_accuracy: 0.5333 - val_loss: 0.7313 - learning_rate: 1.0000e-05
Epoch 4/12
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.5952 - loss: 0.9130 - val_accuracy: 0.5333 - val_loss: 0.7340 - learning_rate: 1.0000e-05
Epoch 5/12
11/11 ━━━━━━━━━━━━━━━━━━━━ 15s 1s/step - accuracy: 0.5119 - loss: 0.9471 - val_accuracy: 0.5333 - val_loss: 0.7366 - learning_rate: 1.0000e-05
Epoch 6/12
11/11 ━━━━━━━━━━━━━━━━━━━━ 16s 1s/step - accuracy: 0.5952 - loss: 0.9659 - val_accuracy: 0.5333 - val_loss: 0.7415 - learning_rate: 1.0000e-05
Epoch 7/12
11/11 ━━━━━━━━━━━━━━━━━━━━ 20s 1s/step - accuracy: 0.6012 - lo

In [25]:
test_loss, test_acc = model.evaluate(test_gen)
print("✅ Dental Model Test Accuracy:", test_acc)

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 480ms/step - accuracy: 0.5484 - loss: 0.7181
✅ Dental Model Test Accuracy: 0.5483871102333069
